In [ ]:
import re

import pandas as pd

from geofeatureviz.io import loader, path_settings
from geofeatureviz.rendering import (
    DiagonalStripedPattern,
    MapSVG,
    OrthoMapSVG,
    map_style,
)
from geofeatureviz_scripts import loader as scripts_loader

RESULT_DIR = path_settings.results_dir / "regional_groups"

In [ ]:
countries = scripts_loader.load_and_prep_data("country", "ne", resolution=110)

regional_groups = loader.load_regional_groups()
country_transl = loader.load_country_translations()

In [ ]:
def get_country_codes(country_names):
    """Get the country code for each country name.

    Args:
        country_names: List of country names for which the codes are requested.

    Returns:
        A list of country codes.
    """
    country_codes = []
    for c in country_names:
        country_code = country_transl[country_transl["german"] == c]["code"].iloc[0]
        country_codes.append(country_code)
    assert len(country_names) == len(country_codes)
    return country_codes


def get_group_id(group_name):
    """Get the ID of a group based on the group name.

    Args:
        group_name: Name of the group.

    Returns:
        A unique identifier of the group that "simplifies" the name, e.g. making it
        lower case, removing umlauts, etc.
    """
    group_id = group_name.lower()
    umlaut_map = str.maketrans({"ä": "ae", "ö": "oe", "ü": "ue", "ß": "ss"})
    group_id = group_id.translate(umlaut_map)
    group_id = re.sub(r"[^a-zA-Z0-9._-]+", "_", group_id)
    return group_id


land_kwargs = map_style.STYLES["land"].copy()
land_kwargs["stroke_width"] = land_kwargs.get("stroke_width", 1) / 2
land_fill = land_kwargs.pop("fill")
highlight_fill = map_style.COLORS["highlight"]
optional_fill = "url(#optional-country-pattern)"
optional_pattern = DiagonalStripedPattern(
    id="optional-country-pattern",
    stripe_colors=(land_fill, highlight_fill),
    stripe_width=(2, 2),
)

iso_cols = [c for c in countries.columns if ("iso_a3" in c) or ("adm0_a3" in c)]

# get countries belonging to a regional group
for reg_group_name, reg_group in regional_groups.items():
    reg_group_id = get_group_id(reg_group_name)

    # get regional group core countries
    reg_country_codes = get_country_codes(reg_group["core"])
    reg_mask = countries[iso_cols].isin(reg_country_codes).any(axis=1)
    reg_countries = countries[reg_mask]
    # get regional group optional countries
    opt_country_codes = get_country_codes(reg_group["optional"])
    opt_mask = countries[iso_cols].isin(opt_country_codes).any(axis=1)
    opt_countries = countries[opt_mask]
    # get all other countries
    other_countries = countries[~(reg_mask | opt_mask)]

    # determine center of regional group
    center = reg_countries.union_all().centroid

    if reg_group["projection"] == "ortho":
        # draw SVG canvas
        canvas = OrthoMapSVG(width=500, center=(center.x, center.y))
        canvas.add_sea()

        # other countries
        canvas.add_gdf(other_countries, "countries", fill=land_fill, **land_kwargs)
        # highlighted countries
        canvas.add_gdf(reg_countries, reg_group_id, fill=highlight_fill, **land_kwargs)
        # optional striped countries
        canvas.add_def(optional_pattern)
        canvas.add_gdf(
            opt_countries, f"{reg_group_id}_optional", fill=optional_fill, **land_kwargs
        )
        canvas.add_shadow(identifier="globeShadow")
    else:
        # draw SVG canvas
        canvas = MapSVG(width=500)
        canvas.add_background(str(map_style.COLORS["lake"]))
        # other countries
        canvas.add_gdf(other_countries, "countries", fill=land_fill, **land_kwargs)
        # highlighted countries
        canvas.add_gdf(reg_countries, reg_group_id, fill=highlight_fill, **land_kwargs)
        # optional striped countries
        canvas.add_def(optional_pattern)
        canvas.add_gdf(
            opt_countries, f"{reg_group_id}_optional", fill=optional_fill, **land_kwargs
        )

    # save svg
    canvas.save(RESULT_DIR / "original" / f"{reg_group_id}.svg")
    # save optimized
    canvas.save(RESULT_DIR / "optimized" / f"{reg_group_id}.svg", optimize=True)

In [ ]:
# create a csv file that can be imported by anki
reg_group_dicts = []
for reg_group_name, reg_group in regional_groups.items():
    reg_group_id = get_group_id(reg_group_name)

    core, optional = reg_group["core"], reg_group["optional"]
    core_str, optional_str = ", ".join(core), ", ".join(optional)
    if optional_str != "":
        country_str = f"{core_str}, ({optional_str})"
    else:
        country_str = core_str

    file_name = f'<img src="{reg_group_id}.svg">'

    reg_group_dicts.append(
        {"Name": reg_group_name, "Countries": country_str, "Map": file_name}
    )
df = pd.DataFrame(reg_group_dicts)

df.to_csv(RESULT_DIR / "regional_groups.csv", index=False, header=False)
df